# CDS Raw Market Data Analysis

This notebook provides a comprehensive analysis of raw CDS market data across different maturities (1Y, 3Y, 5Y), including:
- Summary statistics for each company and maturity
- Missing data analysis
- Sequential repeat detection (data staleness)
- Liquidity metrics

Author: Analysis Date: 2024

## 1. Import Required Libraries

Import necessary libraries for data analysis and visualization.

In [ ]:
# Import Required Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')

# Set visualization style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.4f}'.format)

print("Libraries imported successfully!")

## 2. Load CDS Data

Load the CDS data from Excel files for all three maturities (1Y, 3Y, 5Y).

In [ ]:
# Define data paths
data_path = Path('/Users/thomasdeleeuw/Downloads/Seminar-QF/Seminar QF/data')

# Load CDS data for different maturities
cds_1y = pd.read_excel(data_path / 'CDS_1y_mat_data.xlsx', index_col=0, parse_dates=True)
cds_3y = pd.read_excel(data_path / 'CDS_3y_mat_data.xlsx', index_col=0, parse_dates=True)
cds_5y = pd.read_excel(data_path / 'CDS_5y_mat_data.xlsx', index_col=0, parse_dates=True)

print("CDS Data Loaded Successfully!")
print(f"\n1-Year Maturity: {cds_1y.shape[0]} dates, {cds_1y.shape[1]} companies")
print(f"3-Year Maturity: {cds_3y.shape[0]} dates, {cds_3y.shape[1]} companies")
print(f"5-Year Maturity: {cds_5y.shape[0]} dates, {cds_5y.shape[1]} companies")

# Display first few rows
print("\n1-Year Maturity Data (first 5 rows):")
display(cds_1y.head())

## 3. Summary Statistics by Company and Maturity

Calculate comprehensive summary statistics for each company across all maturities.

In [ ]:
# Function to calculate comprehensive summary statistics
def calculate_summary_stats(df, maturity_name):
    """
    Calculate summary statistics for each company in the dataframe.
    
    Parameters:
    - df: DataFrame with CDS spreads
    - maturity_name: String identifier for the maturity
    
    Returns:
    - DataFrame with summary statistics
    """
    stats_dict = {}
    
    for company in df.columns:
        company_data = df[company].dropna()
        
        if len(company_data) > 0:
            stats_dict[company] = {
                'Count': len(company_data),
                'Mean': company_data.mean(),
                'Median': company_data.median(),
                'Std': company_data.std(),
                'Min': company_data.min(),
                'Max': company_data.max(),
                'Q25': company_data.quantile(0.25),
                'Q75': company_data.quantile(0.75),
                'Range': company_data.max() - company_data.min(),
                'CV': (company_data.std() / company_data.mean()) if company_data.mean() != 0 else np.nan,
                'Skewness': company_data.skew(),
                'Kurtosis': company_data.kurtosis()
            }
        else:
            stats_dict[company] = {key: np.nan for key in ['Count', 'Mean', 'Median', 'Std', 'Min', 'Max', 'Q25', 'Q75', 'Range', 'CV', 'Skewness', 'Kurtosis']}
    
    stats_df = pd.DataFrame(stats_dict).T
    stats_df.index.name = 'Company'
    
    return stats_df

# Calculate summary statistics for all maturities
print("="*80)
print("SUMMARY STATISTICS BY COMPANY AND MATURITY")
print("="*80)

stats_1y = calculate_summary_stats(cds_1y, '1Y')
stats_3y = calculate_summary_stats(cds_3y, '3Y')
stats_5y = calculate_summary_stats(cds_5y, '5Y')

print("\n### 1-YEAR MATURITY SUMMARY STATISTICS ###")
display(stats_1y)

print("\n### 3-YEAR MATURITY SUMMARY STATISTICS ###")
display(stats_3y)

print("\n### 5-YEAR MATURITY SUMMARY STATISTICS ###")
display(stats_5y)

In [ ]:
# Export summary statistics to Excel
output_path = Path('/Users/thomasdeleeuw/Downloads/Seminar-QF/Seminar QF/results')
output_path.mkdir(parents=True, exist_ok=True)

with pd.ExcelWriter(output_path / 'cds_summary_statistics.xlsx') as writer:
    stats_1y.to_excel(writer, sheet_name='1Y_Maturity')
    stats_3y.to_excel(writer, sheet_name='3Y_Maturity')
    stats_5y.to_excel(writer, sheet_name='5Y_Maturity')

print("Summary statistics exported to: cds_summary_statistics.xlsx")

## 4. Missing Data Analysis

Analyze missing values (NAs) for each company at each maturity level.

In [ ]:
# Function to analyze missing data
def analyze_missing_data(df, maturity_name):
    """
    Analyze missing data patterns in the CDS spreads.
    
    Parameters:
    - df: DataFrame with CDS spreads
    - maturity_name: String identifier for the maturity
    
    Returns:
    - DataFrame with missing data analysis
    """
    missing_dict = {}
    total_observations = len(df)
    
    for company in df.columns:
        na_count = df[company].isna().sum()
        na_percentage = (na_count / total_observations) * 100
        first_valid = df[company].first_valid_index()
        last_valid = df[company].last_valid_index()
        
        missing_dict[company] = {
            'Total_Observations': total_observations,
            'NA_Count': na_count,
            'Non_NA_Count': total_observations - na_count,
            'NA_Percentage': na_percentage,
            'Data_Availability': 100 - na_percentage,
            'First_Valid_Date': first_valid,
            'Last_Valid_Date': last_valid
        }
    
    missing_df = pd.DataFrame(missing_dict).T
    missing_df.index.name = 'Company'
    
    return missing_df

# Analyze missing data for all maturities
print("="*80)
print("MISSING DATA ANALYSIS BY COMPANY AND MATURITY")
print("="*80)

missing_1y = analyze_missing_data(cds_1y, '1Y')
missing_3y = analyze_missing_data(cds_3y, '3Y')
missing_5y = analyze_missing_data(cds_5y, '5Y')

print("\n### 1-YEAR MATURITY - MISSING DATA ###")
display(missing_1y.sort_values('NA_Percentage', ascending=False))

print("\n### 3-YEAR MATURITY - MISSING DATA ###")
display(missing_3y.sort_values('NA_Percentage', ascending=False))

print("\n### 5-YEAR MATURITY - MISSING DATA ###")
display(missing_5y.sort_values('NA_Percentage', ascending=False))

In [ ]:
# Visualize missing data patterns
fig, axes = plt.subplots(1, 3, figsize=(20, 8))

for idx, (df, maturity, ax) in enumerate(zip([cds_1y, cds_3y, cds_5y], 
                                               ['1Y', '3Y', '5Y'], 
                                               axes)):
    # Calculate missing percentage for each company
    missing_pct = (df.isna().sum() / len(df)) * 100
    missing_pct = missing_pct.sort_values(ascending=False)
    
    # Plot
    missing_pct.plot(kind='bar', ax=ax, color='coral')
    ax.set_title(f'Missing Data Percentage - {maturity} Maturity', fontsize=14, fontweight='bold')
    ax.set_xlabel('Company', fontsize=12)
    ax.set_ylabel('Missing Data (%)', fontsize=12)
    ax.axhline(y=20, color='red', linestyle='--', alpha=0.5, label='20% threshold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Rotate x-axis labels
    ax.tick_params(axis='x', rotation=90)

plt.tight_layout()
plt.savefig(output_path / 'missing_data_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nMissing data visualization saved!")

## 5. Sequential Repeat Detection

Detect and count consecutive identical CDS spread values (data staleness indicator).

In [ ]:
# Function to detect sequential repeats
def detect_sequential_repeats(df, maturity_name):
    """
    Detect consecutive identical values in CDS spreads (stale data indicator).
    
    Parameters:
    - df: DataFrame with CDS spreads
    - maturity_name: String identifier for the maturity
    
    Returns:
    - DataFrame with sequential repeat analysis
    """
    repeat_dict = {}
    
    for company in df.columns:
        company_data = df[company].dropna()
        
        if len(company_data) > 1:
            # Identify where values change
            value_changes = company_data != company_data.shift(1)
            
            # Create groups of consecutive identical values
            groups = value_changes.cumsum()
            
            # Calculate repeat sequences
            repeat_lengths = company_data.groupby(groups).size()
            
            # Get sequences with more than 1 occurrence (actual repeats)
            actual_repeats = repeat_lengths[repeat_lengths > 1]
            
            repeat_dict[company] = {
                'Total_Valid_Observations': len(company_data),
                'Total_Repeat_Sequences': len(actual_repeats),
                'Total_Repeated_Days': actual_repeats.sum() if len(actual_repeats) > 0 else 0,
                'Max_Consecutive_Repeats': actual_repeats.max() if len(actual_repeats) > 0 else 0,
                'Mean_Repeat_Length': actual_repeats.mean() if len(actual_repeats) > 0 else 0,
                'Median_Repeat_Length': actual_repeats.median() if len(actual_repeats) > 0 else 0,
                'Repeat_Percentage': (actual_repeats.sum() / len(company_data) * 100) if len(actual_repeats) > 0 else 0,
                'Update_Frequency': (len(repeat_lengths) / len(company_data) * 100) if len(company_data) > 0 else 0
            }
        else:
            repeat_dict[company] = {key: 0 for key in ['Total_Valid_Observations', 'Total_Repeat_Sequences', 
                                                        'Total_Repeated_Days', 'Max_Consecutive_Repeats',
                                                        'Mean_Repeat_Length', 'Median_Repeat_Length',
                                                        'Repeat_Percentage', 'Update_Frequency']}
    
    repeat_df = pd.DataFrame(repeat_dict).T
    repeat_df.index.name = 'Company'
    
    return repeat_df

# Detect sequential repeats for all maturities
print("="*80)
print("SEQUENTIAL REPEAT DETECTION (DATA STALENESS ANALYSIS)")
print("="*80)

repeats_1y = detect_sequential_repeats(cds_1y, '1Y')
repeats_3y = detect_sequential_repeats(cds_3y, '3Y')
repeats_5y = detect_sequential_repeats(cds_5y, '5Y')

print("\n### 1-YEAR MATURITY - SEQUENTIAL REPEATS ###")
display(repeats_1y.sort_values('Repeat_Percentage', ascending=False))

print("\n### 3-YEAR MATURITY - SEQUENTIAL REPEATS ###")
display(repeats_3y.sort_values('Repeat_Percentage', ascending=False))

print("\n### 5-YEAR MATURITY - SEQUENTIAL REPEATS ###")
display(repeats_5y.sort_values('Repeat_Percentage', ascending=False))

In [ ]:
# Visualize sequential repeats
fig, axes = plt.subplots(2, 3, figsize=(20, 12))

maturities = ['1Y', '3Y', '5Y']
repeat_dfs = [repeats_1y, repeats_3y, repeats_5y]

for idx, (repeat_df, maturity) in enumerate(zip(repeat_dfs, maturities)):
    # Plot 1: Repeat Percentage
    ax1 = axes[0, idx]
    repeat_pct = repeat_df['Repeat_Percentage'].sort_values(ascending=False)
    repeat_pct.plot(kind='bar', ax=ax1, color='steelblue')
    ax1.set_title(f'Stale Data Percentage - {maturity}', fontsize=12, fontweight='bold')
    ax1.set_xlabel('Company', fontsize=10)
    ax1.set_ylabel('Repeat Percentage (%)', fontsize=10)
    ax1.tick_params(axis='x', rotation=90)
    ax1.grid(True, alpha=0.3)
    
    # Plot 2: Max Consecutive Repeats
    ax2 = axes[1, idx]
    max_repeats = repeat_df['Max_Consecutive_Repeats'].sort_values(ascending=False)
    max_repeats.plot(kind='bar', ax=ax2, color='indianred')
    ax2.set_title(f'Max Consecutive Identical Values - {maturity}', fontsize=12, fontweight='bold')
    ax2.set_xlabel('Company', fontsize=10)
    ax2.set_ylabel('Max Consecutive Days', fontsize=10)
    ax2.tick_params(axis='x', rotation=90)
    ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(output_path / 'sequential_repeats_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nSequential repeats visualization saved!")

## 6. Liquidity Metrics Calculation

Calculate comprehensive liquidity metrics combining data availability, update frequency, and spread volatility.

In [ ]:
# Function to calculate liquidity metrics
def calculate_liquidity_metrics(df, missing_df, repeat_df, maturity_name):
    """
    Calculate comprehensive liquidity metrics for CDS spreads.
    
    Liquidity Score Components:
    1. Data Availability Ratio (0-100): Percentage of non-missing data
    2. Update Frequency Score (0-100): Based on how often data changes (inverse of staleness)
    3. Spread Volatility Score (0-100): Normalized volatility (higher = more liquid)
    4. Composite Liquidity Score (0-100): Weighted average of above metrics
    
    Parameters:
    - df: DataFrame with CDS spreads
    - missing_df: DataFrame with missing data analysis
    - repeat_df: DataFrame with sequential repeat analysis
    - maturity_name: String identifier for the maturity
    
    Returns:
    - DataFrame with liquidity metrics
    """
    liquidity_dict = {}
    
    for company in df.columns:
        # Component 1: Data Availability (from missing data analysis)
        data_availability = missing_df.loc[company, 'Data_Availability']
        
        # Component 2: Update Frequency Score (inverse of repeat percentage)
        repeat_pct = repeat_df.loc[company, 'Repeat_Percentage']
        update_frequency_score = 100 - repeat_pct
        
        # Component 3: Spread Volatility Score
        company_data = df[company].dropna()
        if len(company_data) > 1:
            # Calculate daily returns volatility
            returns = company_data.pct_change().dropna()
            volatility = returns.std() * np.sqrt(252)  # Annualized
            
            # Normalize volatility (higher volatility = higher score, up to a point)
            # Use a sigmoid-like transformation
            volatility_score = min(100, volatility * 1000)  # Scale factor
        else:
            volatility = 0
            volatility_score = 0
        
        # Component 4: Bid-Ask Proxy (using daily range as percentage of level)
        if len(company_data) > 1:
            daily_changes = company_data.diff().abs().dropna()
            avg_daily_change = daily_changes.mean()
            avg_level = company_data.mean()
            daily_range_pct = (avg_daily_change / avg_level * 100) if avg_level != 0 else 0
        else:
            daily_range_pct = 0
        
        # Composite Liquidity Score (weighted average)
        # Weights: Data Availability (40%), Update Frequency (30%), Volatility (20%), Range (10%)
        composite_score = (
            0.40 * data_availability +
            0.30 * update_frequency_score +
            0.20 * min(volatility_score, 100) +
            0.10 * min(daily_range_pct * 10, 100)
        )
        
        liquidity_dict[company] = {
            'Data_Availability_Score': data_availability,
            'Update_Frequency_Score': update_frequency_score,
            'Volatility_Score': min(volatility_score, 100),
            'Annualized_Volatility': volatility * 100,  # In percentage
            'Avg_Daily_Change_bps': avg_daily_change if len(company_data) > 1 else 0,
            'Daily_Range_Pct': daily_range_pct,
            'Composite_Liquidity_Score': composite_score,
            'Liquidity_Category': 'High' if composite_score >= 70 else ('Medium' if composite_score >= 50 else 'Low')
        }
    
    liquidity_df = pd.DataFrame(liquidity_dict).T
    liquidity_df.index.name = 'Company'
    
    return liquidity_df

# Calculate liquidity metrics for all maturities
print("="*80)
print("LIQUIDITY METRICS ANALYSIS")
print("="*80)

liquidity_1y = calculate_liquidity_metrics(cds_1y, missing_1y, repeats_1y, '1Y')
liquidity_3y = calculate_liquidity_metrics(cds_3y, missing_3y, repeats_3y, '3Y')
liquidity_5y = calculate_liquidity_metrics(cds_5y, missing_5y, repeats_5y, '5Y')

print("\n### 1-YEAR MATURITY - LIQUIDITY METRICS ###")
display(liquidity_1y.sort_values('Composite_Liquidity_Score', ascending=False))

print("\n### 3-YEAR MATURITY - LIQUIDITY METRICS ###")
display(liquidity_3y.sort_values('Composite_Liquidity_Score', ascending=False))

print("\n### 5-YEAR MATURITY - LIQUIDITY METRICS ###")
display(liquidity_5y.sort_values('Composite_Liquidity_Score', ascending=False))

In [ ]:
# Export all analysis results to Excel
with pd.ExcelWriter(output_path / 'cds_complete_analysis.xlsx') as writer:
    # Summary Statistics
    stats_1y.to_excel(writer, sheet_name='Summary_Stats_1Y')
    stats_3y.to_excel(writer, sheet_name='Summary_Stats_3Y')
    stats_5y.to_excel(writer, sheet_name='Summary_Stats_5Y')
    
    # Missing Data
    missing_1y.to_excel(writer, sheet_name='Missing_Data_1Y')
    missing_3y.to_excel(writer, sheet_name='Missing_Data_3Y')
    missing_5y.to_excel(writer, sheet_name='Missing_Data_5Y')
    
    # Sequential Repeats
    repeats_1y.to_excel(writer, sheet_name='Repeats_1Y')
    repeats_3y.to_excel(writer, sheet_name='Repeats_3Y')
    repeats_5y.to_excel(writer, sheet_name='Repeats_3Y')
    
    # Liquidity Metrics
    liquidity_1y.to_excel(writer, sheet_name='Liquidity_1Y')
    liquidity_3y.to_excel(writer, sheet_name='Liquidity_3Y')
    liquidity_5y.to_excel(writer, sheet_name='Liquidity_5Y')

print("Complete analysis exported to: cds_complete_analysis.xlsx")

## 7. Visualization of Data Quality

Create comprehensive visualizations including heatmaps and comparative charts.

In [ ]:
# Create comprehensive liquidity comparison visualization
fig, axes = plt.subplots(2, 2, figsize=(20, 16))

# Plot 1: Composite Liquidity Score Comparison
ax1 = axes[0, 0]
liquidity_comparison = pd.DataFrame({
    '1Y': liquidity_1y['Composite_Liquidity_Score'],
    '3Y': liquidity_3y['Composite_Liquidity_Score'],
    '5Y': liquidity_5y['Composite_Liquidity_Score']
})
liquidity_comparison.plot(kind='bar', ax=ax1, width=0.8)
ax1.set_title('Composite Liquidity Score by Maturity', fontsize=14, fontweight='bold')
ax1.set_xlabel('Company', fontsize=12)
ax1.set_ylabel('Liquidity Score (0-100)', fontsize=12)
ax1.legend(title='Maturity', fontsize=10)
ax1.tick_params(axis='x', rotation=90)
ax1.axhline(y=70, color='green', linestyle='--', alpha=0.5, label='High Liquidity')
ax1.axhline(y=50, color='orange', linestyle='--', alpha=0.5, label='Medium Liquidity')
ax1.grid(True, alpha=0.3)

# Plot 2: Liquidity Components Heatmap (5Y as example)
ax2 = axes[0, 1]
liquidity_components = liquidity_5y[['Data_Availability_Score', 'Update_Frequency_Score', 
                                      'Volatility_Score', 'Composite_Liquidity_Score']].T
sns.heatmap(liquidity_components, annot=True, fmt='.1f', cmap='RdYlGn', ax=ax2, 
            cbar_kws={'label': 'Score (0-100)'}, vmin=0, vmax=100)
ax2.set_title('Liquidity Components Heatmap - 5Y Maturity', fontsize=14, fontweight='bold')
ax2.set_xlabel('Company', fontsize=12)
ax2.set_ylabel('Liquidity Component', fontsize=12)

# Plot 3: Data Availability vs Update Frequency
ax3 = axes[1, 0]
for maturity, liquidity_df, color in zip(['1Y', '3Y', '5Y'], 
                                          [liquidity_1y, liquidity_3y, liquidity_5y],
                                          ['blue', 'green', 'red']):
    ax3.scatter(liquidity_df['Data_Availability_Score'], 
               liquidity_df['Update_Frequency_Score'],
               s=100, alpha=0.6, label=maturity, color=color)

ax3.set_title('Data Availability vs Update Frequency', fontsize=14, fontweight='bold')
ax3.set_xlabel('Data Availability Score', fontsize=12)
ax3.set_ylabel('Update Frequency Score', fontsize=12)
ax3.legend(title='Maturity', fontsize=10)
ax3.grid(True, alpha=0.3)

# Plot 4: Liquidity Category Distribution
ax4 = axes[1, 1]
liquidity_categories = pd.DataFrame({
    '1Y': liquidity_1y['Liquidity_Category'].value_counts(),
    '3Y': liquidity_3y['Liquidity_Category'].value_counts(),
    '5Y': liquidity_5y['Liquidity_Category'].value_counts()
}).T.fillna(0)

liquidity_categories.plot(kind='bar', stacked=True, ax=ax4, 
                          color=['red', 'orange', 'green'])
ax4.set_title('Liquidity Category Distribution by Maturity', fontsize=14, fontweight='bold')
ax4.set_xlabel('Maturity', fontsize=12)
ax4.set_ylabel('Number of Companies', fontsize=12)
ax4.legend(title='Liquidity Category', fontsize=10)
ax4.tick_params(axis='x', rotation=0)
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(output_path / 'liquidity_comprehensive_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nComprehensive liquidity visualization saved!")

In [ ]:
# Create missing data heatmap across all maturities
fig, axes = plt.subplots(3, 1, figsize=(16, 12))

for idx, (df, maturity, ax) in enumerate(zip([cds_1y, cds_3y, cds_5y], 
                                               ['1Y', '3Y', '5Y'], 
                                               axes)):
    # Create binary matrix of missing data (sample every 20th row for visibility)
    sample_freq = max(1, len(df) // 50)
    missing_matrix = df.iloc[::sample_freq].isna().T
    
    sns.heatmap(missing_matrix, cmap='RdYlGn_r', cbar=True, ax=ax,
                xticklabels=False, yticklabels=True)
    ax.set_title(f'Missing Data Pattern - {maturity} Maturity', fontsize=14, fontweight='bold')
    ax.set_xlabel('Time Period (sampled)', fontsize=12)
    ax.set_ylabel('Company', fontsize=12)

plt.tight_layout()
plt.savefig(output_path / 'missing_data_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nMissing data heatmap saved!")

## Summary and Key Findings

Generate a summary report of key findings from the data quality analysis.

In [ ]:
# Generate summary report
print("="*80)
print("SUMMARY REPORT: CDS RAW DATA QUALITY ANALYSIS")
print("="*80)

for maturity, stats_df, missing_df, repeat_df, liquidity_df in zip(
    ['1-Year', '3-Year', '5-Year'],
    [stats_1y, stats_3y, stats_5y],
    [missing_1y, missing_3y, missing_5y],
    [repeats_1y, repeats_3y, repeats_5y],
    [liquidity_1y, liquidity_3y, liquidity_5y]
):
    print(f"\n{'='*80}")
    print(f"{maturity} Maturity Summary")
    print(f"{'='*80}")
    
    print(f"\n1. Data Availability:")
    print(f"   - Best coverage: {missing_df['Data_Availability'].idxmax()} "
          f"({missing_df['Data_Availability'].max():.2f}%)")
    print(f"   - Worst coverage: {missing_df['Data_Availability'].idxmin()} "
          f"({missing_df['Data_Availability'].min():.2f}%)")
    print(f"   - Average coverage: {missing_df['Data_Availability'].mean():.2f}%")
    
    print(f"\n2. Data Staleness (Sequential Repeats):")
    print(f"   - Most stale: {repeat_df['Repeat_Percentage'].idxmax()} "
          f"({repeat_df['Repeat_Percentage'].max():.2f}%)")
    print(f"   - Least stale: {repeat_df['Repeat_Percentage'].idxmin()} "
          f"({repeat_df['Repeat_Percentage'].min():.2f}%)")
    print(f"   - Average staleness: {repeat_df['Repeat_Percentage'].mean():.2f}%")
    
    print(f"\n3. Liquidity Metrics:")
    print(f"   - Most liquid: {liquidity_df['Composite_Liquidity_Score'].idxmax()} "
          f"(Score: {liquidity_df['Composite_Liquidity_Score'].max():.2f})")
    print(f"   - Least liquid: {liquidity_df['Composite_Liquidity_Score'].idxmin()} "
          f"(Score: {liquidity_df['Composite_Liquidity_Score'].min():.2f})")
    print(f"   - Average liquidity score: {liquidity_df['Composite_Liquidity_Score'].mean():.2f}")
    
    # Liquidity distribution
    high_liquidity = (liquidity_df['Liquidity_Category'] == 'High').sum()
    medium_liquidity = (liquidity_df['Liquidity_Category'] == 'Medium').sum()
    low_liquidity = (liquidity_df['Liquidity_Category'] == 'Low').sum()
    
    print(f"\n4. Liquidity Distribution:")
    print(f"   - High liquidity companies: {high_liquidity}")
    print(f"   - Medium liquidity companies: {medium_liquidity}")
    print(f"   - Low liquidity companies: {low_liquidity}")
    
    print(f"\n5. Spread Statistics:")
    print(f"   - Highest average spread: {stats_df['Mean'].idxmax()} "
          f"({stats_df['Mean'].max():.2f} bps)")
    print(f"   - Lowest average spread: {stats_df['Mean'].idxmin()} "
          f"({stats_df['Mean'].min():.2f} bps)")
    print(f"   - Most volatile: {stats_df['Std'].idxmax()} "
          f"(Std: {stats_df['Std'].max():.2f} bps)")

print("\n" + "="*80)
print("Analysis complete! All results have been exported.")
print("="*80)

## Conclusions

### Liquidity Metric Methodology

The **Composite Liquidity Score** is calculated as a weighted average of four components:

1. **Data Availability Score (40% weight)**: Percentage of non-missing observations
   - Higher score = more complete data coverage

2. **Update Frequency Score (30% weight)**: Inverse of the sequential repeat percentage
   - Higher score = data updates more frequently (less stale)

3. **Volatility Score (20% weight)**: Normalized annualized volatility
   - Higher volatility suggests more active trading and price discovery

4. **Daily Range Score (10% weight)**: Average daily spread changes relative to level
   - Higher range suggests better price discovery and liquidity

### Interpretation Guidelines

- **High Liquidity** (Score ≥ 70): Good data quality, frequent updates, actively traded
- **Medium Liquidity** (50 ≤ Score < 70): Acceptable quality with some limitations
- **Low Liquidity** (Score < 50): Poor data quality, stale prices, illiquid market

### Recommendations

1. Use companies with **High Liquidity** scores for primary analysis
2. Apply caution when using data with high sequential repeat percentages
3. Consider excluding companies with >20% missing data
4. The 5-year maturity typically shows the best liquidity across most names